# Lakehouse Files folder hotspot inventory

This notebook inventories exclusively the `Files` area of the attached default Lakehouse.

It uses the OneLake ADLS-compatible **recursive Path List API**, which returns up to 5,000 paths per request. File contents are never read.

The output highlights:

- largest folders by physical file size;
- file counts and average file size;
- small-file hotspots that can increase metadata and Spark processing costs.

> Start with `MAX_DEPTH = 1` or `2`. After identifying a large branch, set `ROOT_SUBPATH` to that folder and rerun the scan for a focused drill-down.


In [ ]:
# Scan parameters
ROOT_SUBPATH = ""          # Leave empty for Files root, or use a relative path such as "archive/country=FR"
MAX_DEPTH = 2             # Number of directory levels to aggregate
TOP_N = 200
PAGE_SIZE = 5000          # OneLake API maximum page size
SMALL_FILE_MB = 16
MAX_RETRIES = 8
REQUEST_TIMEOUT_SECONDS = 120
PROGRESS_EVERY_PAGES = 100


In [ ]:
from collections import defaultdict
import time

import requests
from pyspark.sql import functions as F


context = notebookutils.runtime.context
workspace_id = context["defaultLakehouseWorkspaceId"]
lakehouse_id = context["defaultLakehouseId"]

if not workspace_id or not lakehouse_id:
    raise RuntimeError("Attach a default Lakehouse to the notebook before running the inventory.")

endpoint = notebookutils.conf.get("trident.onelake.endpoint").rstrip("/")
if not endpoint.startswith("http"):
    endpoint = f"https://{endpoint}"

filesystem_url = f"{endpoint}/{workspace_id}"


def normalize_files_subpath(value):
    # Accept common Fabric mount-style inputs but convert them to a path
    # relative to the OneLake Lakehouse/Files root used by the REST API.
    subpath = (value or "").strip().replace("\\", "/").strip("/")
    lakehouse_files = f"{lakehouse_id}/Files"

    for prefix in (lakehouse_files, "lakehouse/default/Files", "Files"):
        if subpath == prefix:
            return ""
        if subpath.startswith(prefix + "/"):
            return subpath[len(prefix) + 1:]

    return subpath


relative_subpath = normalize_files_subpath(ROOT_SUBPATH)
root_directory = f"{lakehouse_id}/Files"

if relative_subpath:
    root_directory += f"/{relative_subpath}"

if ROOT_SUBPATH and ROOT_SUBPATH.strip("/") != relative_subpath:
    print(f"Normalized ROOT_SUBPATH '{ROOT_SUBPATH}' to '{relative_subpath or '(Files root)'}'.")

root_prefix = root_directory.rstrip("/") + "/"
small_file_bytes = SMALL_FILE_MB * 1024**2


def empty_stats():
    return {
        "size_bytes": 0,
        "file_count": 0,
        "small_file_count": 0,
    }


stats = defaultdict(empty_stats)
session = requests.Session()
access_token = notebookutils.credentials.getToken("storage")


def request_page(params):
    global access_token

    for attempt in range(MAX_RETRIES):
        headers = {
            "Authorization": f"Bearer {access_token}",
            "x-ms-version": "2023-08-03",
        }

        response = session.get(
            filesystem_url,
            headers=headers,
            params=params,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )

        if response.status_code == 401:
            access_token = notebookutils.credentials.getToken("storage")
            continue

        if response.status_code == 429 or 500 <= response.status_code < 600:
            retry_after = response.headers.get("Retry-After")
            delay = int(retry_after) if retry_after and retry_after.isdigit() else min(60, 2**attempt)
            print(f"OneLake returned {response.status_code}; retrying in {delay}s.")
            time.sleep(delay)
            continue

        response.raise_for_status()
        return response

    raise RuntimeError(f"OneLake listing failed after {MAX_RETRIES} attempts.")


params = {
    "resource": "filesystem",
    "directory": root_directory,
    "recursive": "true",
    "maxResults": str(PAGE_SIZE),
}

page_count = 0
entry_count = 0
file_count = 0

print(f"Scanning Lakehouse Files only: {root_directory}")
scan_started = time.perf_counter()

while True:
    response = request_page(params)
    entries = response.json().get("paths", [])

    page_count += 1
    entry_count += len(entries)

    for entry in entries:
        if str(entry.get("isDirectory", "false")).lower() == "true":
            continue

        name = entry["name"]
        if not name.startswith(root_prefix):
            continue

        file_count += 1
        relative_path = name[len(root_prefix):].lstrip("/")
        parent_parts = relative_path.split("/")[:-1]
        size = int(entry.get("contentLength") or 0)

        if parent_parts:
            buckets = [
                (depth, "/".join(parent_parts[:depth]))
                for depth in range(1, min(MAX_DEPTH, len(parent_parts)) + 1)
            ]
        else:
            buckets = [(0, "(root files)")]

        for depth, folder in buckets:
            current = stats[(depth, folder)]
            current["size_bytes"] += size
            current["file_count"] += 1
            current["small_file_count"] += int(size < small_file_bytes)

    continuation = response.headers.get("x-ms-continuation")
    if not continuation:
        break

    params["continuation"] = continuation

    if page_count % PROGRESS_EVERY_PAGES == 0:
        elapsed_minutes = (time.perf_counter() - scan_started) / 60
        print(
            f"pages={page_count:,}, entries={entry_count:,}, "
            f"files={file_count:,}, elapsed={elapsed_minutes:,.1f} min"
        )

rows = []

for (depth, folder), values in stats.items():
    count = values["file_count"]
    rows.append(
        (
            depth,
            folder,
            values["size_bytes"],
            values["size_bytes"] / 1024**3,
            values["size_bytes"] / 1024**4,
            count,
            values["size_bytes"] / count / 1024**2 if count else 0.0,
            values["small_file_count"],
            100.0 * values["small_file_count"] / count if count else 0.0,
        )
    )

inventory_schema = """
depth int,
folder string,
size_bytes long,
size_gb double,
size_tb double,
file_count long,
avg_file_mb double,
small_file_count long,
small_file_pct double
"""

inventory = spark.createDataFrame(rows, schema=inventory_schema)
elapsed_minutes = (time.perf_counter() - scan_started) / 60

print(
    f"Completed: {file_count:,} files, {entry_count:,} paths, "
    f"{page_count:,} paged requests, {elapsed_minutes:,.1f} minutes."
)


## Largest folders

Use this view to identify the main physical storage contributors. `depth = 1` represents direct children of the selected root; deeper rows provide drill-down detail.


In [ ]:
display(
    inventory
    .filter(F.col("depth").between(1, MAX_DEPTH))
    .orderBy(F.desc("size_bytes"))
    .limit(TOP_N)
)


## Small-file hotspots

A high file count combined with a small average file size is a compaction candidate. This often reduces metadata operations and improves Spark read performance.


In [ ]:
display(
    inventory
    .filter((F.col("depth") >= 1) & (F.col("small_file_count") > 0))
    .orderBy(F.desc("small_file_count"), F.desc("small_file_pct"))
    .limit(TOP_N)
)


## Interpretation and limitations

- `size_tb`: candidates for retention review, deletion, or archiving.
- `small_file_count` and `small_file_pct`: candidates for file compaction.
- Exact recursive folder sizes always require enumerating every file at least once. The benefit here is that OneLake is scanned once in pages, rather than through one serial `ls()` call per directory.
- The scan reports visible files only. Use **Workspace settings → OneLake → Storage report** for item-level totals including soft-deleted and system data.
- If a path is a shortcut to external storage, its listed data size is not necessarily billable OneLake storage. Inventory the underlying ADLS account when financial attribution is required.

Official references:

- [OneLake API parity](https://learn.microsoft.com/en-us/fabric/onelake/onelake-api-parity)
- [ADLS Path List API](https://learn.microsoft.com/en-us/rest/api/storageservices/datalakestoragegen2/path/list)
- [Get the size of OneLake items](https://learn.microsoft.com/en-us/fabric/onelake/how-to-get-item-size)
